# 第三章 LangChain进阶组件实操
## 3.1 状态管理层(Memory): 让模型拥有记忆能力
### 3.1.1 对话的本质与作用
#### 3.1.1.1 核心本质

In [ ]:
import os 
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI

# 加载环境变量（确保.env文件中配置了API_KEY）
load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")

#初始化LLM模型
llm=ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="mimo-v2.5-pro",
    temperature=0.3 ,
    max_tokens=1000
)

#1.定义提示词模板（包含历史消息占位符）
#创建了一个对话提示模板，它就像是给AI设定了一个“剧本”，规定了对话的结构和角色。
#system是系统提示词，告诉模型应该扮演什么样的角色
full_memory_prompt=ChatPromptTemplate.from_messages( 
    [("system","你是友好的对话助手，需基于完整的历史对话回答用户问题。"),
     MessagesPlaceholder(variable_name="chat_history"), #历史消息占位符
     """
    这是一个历史消息占位符。这是实现对话记忆（记忆功能）的关键。它不在模板中写死任何内容，
    而是在程序运行时，动态地将之前的对话记录（比如用户之前问了什么，AI回答了什么）
    插入到这个位置。这样，AI在回答新问题时就能参考上下文，实现连贯的多轮对话。
    """
     ("human","{user_input}") #用户当前输入
     ])

# 2. 构建基础链（提示词 + LLM）
base_chain = full_memory_prompt | llm

# 3. 会话历史存储（内存模式，生产环境可替换为数据库存储）
full_memory_store = {}

# 4. 定义会话历史获取函数（核心：返回完整历史）
def get_full_memory_history(session_id: str) -> BaseChatMessageHistory:
    """根据session_id获取会话历史，不存在则创建新的历史记录"""
    if session_id not in full_memory_store:
        full_memory_store[session_id] = InMemoryChatMessageHistory()
    return full_memory_store[session_id]

# 5. 构建带全量记忆的对话链
full_memory_chain = RunnableWithMessageHistory(
    runnable=base_chain,
    get_session_history=get_full_memory_history,
    input_messages_key="user_input",  # 输入中用户问题的键名
    history_messages_key="chat_history"  # 传入提示词的历史消息键名 
)

# 测试多轮对话（指定session_id=user_001，隔离不同用户）
config = {"configurable": {"session_id": "user_001"}}

# 第一轮对话
response1 = full_memory_chain.invoke({"user_input": "我叫小明，喜欢编程"}, config=config)
print("助手回复1：", response1.content)

#第二轮对话
response2=full_memory_chain.invoke({"user_input":"我刚才说我喜欢什么？"},config=config)
print("助手回复2：",response2.content)

#查看完整历史记录
print("\n全量记忆的对话历史")
for msg in get_full_memory_history("user_001").messages:
    print(f"{msg.type}:{msg.content}")

助手回复1： 你好呀，小明！很高兴认识你～😊 编程是一个超酷的爱好！你目前在用什么编程语言？或者正在开发什么有趣的项目吗？✨
助手回复2： 你刚才说你喜欢**编程**！😄

你有什么正在开发的项目，或者特别感兴趣的语言/方向吗？

全量记忆的对话历史
human:我叫小明，喜欢编程
ai:你好呀，小明！很高兴认识你～😊 编程是一个超酷的爱好！你目前在用什么编程语言？或者正在开发什么有趣的项目吗？✨
human:我刚才说我喜欢什么？
ai:你刚才说你喜欢**编程**！😄

你有什么正在开发的项目，或者特别感兴趣的语言/方向吗？


#### 3.1.2.2 窗口记忆

In [21]:
#1.定义提示词模板（与全量记忆通用，可复用）
window_memory_prompt=ChatPromptTemplate.from_messages([
    ("system","你是友好的对话助手，需基于最近的对话历史回答用户问题。"),
     MessagesPlaceholder(variable_name="chat_history"),
     ("human","{user_input}")
])

#2.构建基础连
window_base_chain=window_memory_prompt| llm

#3.会话历史存储
window_memory_store={}
WINDOW_SIZE=2 #保留最近2轮对话（即最近4条消息：用户->助手->用户->助手）

#4.定义带窗口限制的会话历史获取函数
def get_window_memory_history(session_id:str)->BaseChatMessageHistory:
    """获取会话历史，仅保留最近WINDOW_SIZE轮对话"""
    if session_id not in window_memory_store:
        window_memory_store[session_id]=InMemoryChatMessageHistory()

    #获取完整历史，截取最近WINDOW_SIZE轮（每轮2条消息）
    history=window_memory_store[session_id]
    #print("history: ",history)
    #print("history.messages:",history.messages)
    if len(history.messages)>2*WINDOW_SIZE:
        #截取后WINDOW_SIZE轮消息（保留最新的）
        history.messages=history.messages[-2*WINDOW_SIZE:]
    return history
    
#5.构建带窗口记忆的对话链
window_memory_chain=RunnableWithMessageHistory(
    runnable=window_base_chain,
    get_session_history=get_window_memory_history,
    input_messages_key="user_input",
    history_messages_key="chat_history",
)

#测试多轮会话（session_id=user_002，与全量记忆会话隔离）
config={"configurable":{"session_id":"user_002"}}

# 模拟5轮对话，验证窗口记忆的截断效果
inputs = [
    "我叫小红",
    "我喜欢画画",
    "我来自上海",
    "我是一名学生",
    "我刚才说我来自哪里？",  # 第5轮：询问第3轮的信息，验证窗口截断
    "我叫什么名字？"  # 第6轮：询问第1轮的信息，验证窗口记忆
]

for i, user_input in enumerate(inputs, 1): #1表示从序号1开始
    response = window_memory_chain.invoke({"user_input": user_input}, config=config)
    print(f"\n第{i}轮 - 助手回复：", response.content)

# 查看窗口记忆的最终历史（仅保留最近2轮）
print("\n窗口记忆的最终对话历史（最近2轮）：")
for msg in get_window_memory_history("user_002").messages:
    print(f"{msg.type}: {msg.content}")



第1轮 - 助手回复： 你好，小红！很高兴认识你！😊 我是你的对话助手，有什么我可以帮你的吗？

第2轮 - 助手回复： 太棒了！🎨 画画是一种非常美好的表达方式，能让人静下心来感受世界。你平时喜欢画什么呢？比如风景、人物、动漫，还是随心涂鸦？

如果愿意的话，可以随时和我分享你的作品或者画画时的心情～我也很想听听你和画画之间的故事呢！(✧ω✧)

第3轮 - 助手回复： 上海真是一个充满艺术气息的城市呢！🌆  
外滩的万国建筑、弄堂里的老街风情、还有各种美术馆和画廊（比如中华艺术宫、西岸美术馆），都特别适合写生或找灵感～

你平时会在上海的哪些地方画画呢？或者有没有特别喜欢的本地艺术家？🎨✨

第4轮 - 助手回复： 学生时代正是培养兴趣爱好的好时候呢！📚✨

你是学美术相关专业的吗？还是平时利用课余时间画画放松一下？

在学校有没有参加美术社团或者画画的兴趣班呀？🎨

第5轮 - 助手回复： 你说的是**上海**呀！🏙️

刚才在对话开头你告诉我的～

第6轮 - 助手回复： 抱歉，你在我们之前的对话中并没有告诉我你的名字呢～😊

我刚才回答你来自上海可能是我记错了，实在抱歉！🙏

你愿意告诉我你叫什么名字吗？

窗口记忆的最终对话历史（最近2轮）：
human: 我刚才说我来自哪里？
ai: 你说的是**上海**呀！🏙️

刚才在对话开头你告诉我的～
human: 我叫什么名字？
ai: 抱歉，你在我们之前的对话中并没有告诉我你的名字呢～😊

我刚才回答你来自上海可能是我记错了，实在抱歉！🙏

你愿意告诉我你叫什么名字吗？


#### 3.1.2.3摘要记忆

In [ ]:
#1. 定义摘要生成提示词（用于压缩对话历史）
summary_prompt=ChatPromptTemplate.from_messages([
    ("system","你是对话摘要助手，需简洁总结以下对话的核心信息（包含用户身份、偏好、关键问题等），不超过50字。"),
    ("human","对话历史: {chat_history_text}\n请生成摘要: ")
])

#2.构建摘要生成链(输入完整历史文本，输出摘要)
summary_chain=summary_prompt|llm

#3.定义对话记忆提示词（注入摘要而非完整历史）
summary_memory_prompt=ChatPromptTemplate.from_messages([
    ("system","你是友好的对话助手，需基于对话摘要回答用户问题，摘要包含核心上下文信息。"),
    ("system","对话摘要: {chat_summary}"), #注入摘要
    ("human","{user_input}")
])

#4.构建基础对话链（提示词+LLM）
"""动态生成chat_summary"""
"""
x就是当前链收到的输入字典
x = {
    "user_input": "我叫什么名字？",
    "chat_history": [
        HumanMessage(content="我叫小红"),
        AIMessage(content="你好小红"),
        HumanMessage(content="我喜欢画画")
    ]}
拿出x["chat_history"]

[f"{msg.type}: {msg.content}" for msg in x["chat_history"]]
变成
[
    "human: 我叫小红",
    "ai: 你好小红",
    "human: 我喜欢画画"
]

"\n".join()变成一个多行字符串:
human: 我叫小红
ai: 你好小红
human: 我喜欢画画
"""
summary_base_chain=(
    RunnablePassthrough.assign( #RunnablePaassthrough:原样保留数据, .assign:在原来输入字典里面，额外添加一个新字段
        chat_summary=lambda x:summary_chain.invoke(
            {
                "chat_history_text":"\n".join(
                    [f"{msg.type}: {msg.content}" for msg in x["chat_history"]]
                )
            }
        ).content #因为summary_chain.invoke(...)返回一个AIMessage,然后content得到比如"用户叫小红,喜欢画画。"放到chat_summary里面
    )
    |summary_memory_prompt #然后传入summary_memory_prompt模板
    |llm
)
"""
输入里有:
{
  "user_input": "我叫什么名字？",
  "chat_history": [...],
  "chat_summary": "用户叫小红，喜欢画画。"
}
然后使用user_input和chat_summary两个字段"""

